In [15]:
from echo.settings import debug_mode
from echo.indexing import create_tables
from echo.runner import make_call
import nest_asyncio
import asyncio


nest_asyncio.apply()

seller = 'https://whatfix.com/'
buyer = 'https://www.rippling.com/'

create_tables(seller)


debug_mode

Created tables for https://whatfix.com/


False

In [ ]:
from echo.step_templates.generic import CallType

call_id = 1
inputs = {
    'seller': seller,
    'call_id': call_id,
    'stakeholders': [
        "Product Manager",
        "Chief Financial Officer",
        "Chief Technology Officer",
        "VP of Sales"
    ]
}

discovery_data = asyncio.run(make_call(call_type=CallType.DISCOVERY.value, clients=[buyer], inputs=inputs))

In [ ]:
inputs['call_id'] = call_id + 1
demo_data = asyncio.run(make_call(call_type=CallType.DEMO.value, clients=[buyer], inputs=inputs))

In [ ]:
inputs['call_id'] = call_id + 2
pricing_data = asyncio.run(make_call(call_type=CallType.PRICING.value, clients=[buyer], inputs=inputs))

In [ ]:
inputs['call_id'] = call_id + 3
negotiation_data = asyncio.run(make_call(call_type=CallType.NEGOTIATION.value, clients=[buyer], inputs=inputs))

In [7]:
from echo.data.indexes import IndexType, IndexDataType
from echo.query_executor import Query, LlamaSubQuery, QueryChain
from echo.step_templates.utilities.account_plan_creation import QueryTypes
from echo.query_executor import arun_query_chain
# from echo.query_executor import aget_query_response
import asyncio

import nest_asyncio
nest_asyncio.apply()


inputs = {
    "seller": seller,
    "buyer": buyer,
}

In [8]:
account_plan = Query(
    query="You're a strategic B2B seller. Based on the following public signals about {buyer}.\n" 
    "Extract 3-5 key initiatives or priorities the company is likely pursuing this year or quarter.\n"
    "Phrase each as a business goal. Do NOT include vague goals. Be specific.\n"
    "Signals for the various initiatives are given below:\n"

    "\nReturn format:\n"
    "- Initiative: clear description\n"
    "- Supporting evidence: source\n",
    sub_queries=[
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors for the account that that client needs to consider?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news for the account?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        )
    ],
    output_name="account_plan"
)

In [9]:
# response = asyncio.run(aget_query_response(echo_query=account_plan, inputs=inputs))
# print(response[0])

In [10]:
account_plan_value_prop = Query(
    query="""
    You are a strategic sales assistant. Given a set of company initiatives and the product profile of the sellers product below as context,
    identify which initiatives are *relevant* to what this product solves.

    For each initiative:
    - Mark as Relevant or Not Relevant
    - If Relevant: explain which product capability maps to it
    - If Not Relevant: explain why it's not a fit (e.g., not adjacent, unrelated)


    output format:
        "initiative": "...",
        "relevant": not_relevant/ mid / highly relevant,
        "mapped_to_product": "...",
        "reasoning": "..."
        "similar buyers and their roi': "...",

    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
        ),
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors that buyer might be worried about and want to tackle",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news and recent media for the buyeraccount?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        ),
        LlamaSubQuery(
            query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value},
        )
    ],
    output_name="account_plan_value_prop",
)

In [11]:
# response = asyncio.run(aget_query_response(echo_query=account_plan_value_prop, inputs=inputs))
# print(response[0])

In [ ]:
competitor_differentiator_value_prop = Query(
    query="""You need to generate a few value propositions and business cases that the seller Whatfix can then use to sell their solution to the buyer Manpower group. 
            The seller is whatfix and the buyer is manpower group
            First identify the top financial, strategic, competitive and priorities evident from news and media to craft top issues and focus points of the buyer.
            Next deeply understand the sellers product, the core problems it has and does solve for its buyers.
            Please make sure to properly align value prop to actual business cases and not just generic value prop. 
            Also understand deeply what the seller sells and the kind of impact it can have before answering. 
            Think deeply
            Now finally, craft a set of value propositions and business cases that the sellers product can solve in alignment with the buyers priorities identified. This will be used by an account executive to pitch the product to the buyer and align with their priorities. so be clear, detailed and specific.
            Use the sellers product info, testimonials, broad initiatives theyve tackled for other customers and how they can align with the buyers strategic, financial and competitive priorities. Also include news and media about the buyer into consideration for further hints and signals on buyer priorities
            """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors that buyer might be worried about and want to tackle",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news and recent media for the buyer account?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        ),
        LlamaSubQuery(
            query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
)

In [13]:
# response = asyncio.run(aget_query_response(echo_query=competitor_differentiator_value_prop, inputs=inputs))
# print(response[0])

In [14]:
# 1) multi threading team gen 
from echo.query_executor import PerplexicaSourceExtraction, PerplexicaSubQuery


multi_threading_team_gen = Query(
    query=(
        "You're an experienced enterprise seller. Given these company initiatives, for the ones marked relevant to the seller's product, "
        "Infer which internal team likely owns or sponsors each initiative. \n"
        "If multiple teams are involved, note primary and secondary.\n"

        "Input:\n"
        "{account_plan_value_prop}\n"

        "Return format:\n"
        "- Initiative: ...\n"
        "- Likely owning team(s): ...\n"
        "- Reasoning:\n"
    ),
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_team_gen",
)

perplexity_search_query = Query(
    query="""
    You are provided with the company initiatives and the owning team for each of them.
    
    {multi_threading_team_gen}.
    
    You are provided with the company initiatives and the relevant information about from the web about the potential team members and employees of the company that would be relevant to the company initiatives.
    Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.
    """,
    sub_queries=[
        PerplexicaSubQuery(
            query=(
                "You are provided with the company initiatives and the owning team for each of them\n"
                "{multi_threading_team_gen}\n"
                "Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.\n"
            ),
            source_extraction_prompts=PerplexicaSourceExtraction(
                system_prompt=(
                    "You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning "
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                ),
                user_prompt=(
                    "You are provided with the company initiatives and the owning team for each of them\n"
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                    "Extract out the team members or employees from the below data\n"
                )
            )
        )
    ],
    output_name="perplexity_search_query"
)

# 2) do a perplexica serach here using team name and buyer name FOR EACH INITIATIVE RETURNED FROM ABOVE
# Search for all possible employees and leaders  from linkedin who belong to that team and extract role, name, and background


# 3) multi threading ROLE AND PERSON EXTRACTOR - use above response also aas input below additionally 

multi_threading_person_extractor = Query(
    query="""
        You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning 
        for every relavant initiative the company is pursuing.
        
        
        You are provided with the company initiatives and the owning team for each of them.
        {multi_threading_team_gen}.
        
        You are further provided with the relevant team members and employees of the company that would be relevant to the company initiatives as crawled from the web.
        {perplexity_search_query}
        
        Do the following:

        Given the company {buyer}, and the owning team of that initiative and the team members crawled from linkedin,
        return people who match titles commonly associated with owning this initiative.
        Focus on seniority, team fit, and tenure. Prioritize those with likely budget/influence.

        Also, Classify each as a champion, decision maker, gatekeeper and influncer within the team responsible for the initiative.
        champion - one who directly owns the pain and will want it solved
        decision maker- the one with power to purchase in the team and for the initiative
        gatekeeper - the one who will block the deal from happening or be tough to convince. This is the only role that could be outside the team like procurement , legal etc.
        influencer - the one who will influence the decision maker and champion to buy the product.

        Return:
        - initiative
        - Name
        - Title
        - Tenure
        - Team
        - Reason they likely own this initiative
        - Classification (champion, decision maker, gatekeeper, influencer) and why

        here is the list of initiatives and the owning team for each of them:
        {multi_threading_team_gen}
    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value} ## Has demo data separately
        )
    ],
    output_name="multi_threading_person_extractor"
)


# 4) multi threading outreach generator

multi_threading_outreach_generator = Query(
    query="""
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in the list
        Do the following:


        Based on this buyer's title, initiative, 
        and recent activity, seller's product details and generate a 1st outreach email that aligns to their business goals and personal context.
        You are also given similar companies the seller has helped before below.

        Tone: Crisp, consultative, relevant.

        Return:
        - Subject line
        - Message body (under 100 words)
        - CTA
        - Persona
        - Title
        - Reasoning for message


        Input:
        {multi_threading_person_extractor}
        
    """,
    sub_queries=[
        LlamaSubQuery(
            query="what is the seller's product and what pains does it solve?",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="what are some companies the sellers product has helped before? be specific and metric driven",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_outreach_generator"
)


In [14]:
# response = asyncio.run(aget_query_response(echo_query=multi_threading_team_gen, inputs=inputs))
# print(response[0])

In [15]:
query_chain = QueryChain(
    queries=[
        account_plan_value_prop,
        multi_threading_team_gen,
        perplexity_search_query,
        multi_threading_person_extractor,
        multi_threading_outreach_generator
    ]
)

In [ ]:
response = asyncio.run(arun_query_chain(query_chain=query_chain, inputs=inputs))

In [ ]:
for r in response.responses:
    print("Query: ", r.query)
    print("Response: ", r.response)

In [ ]:
import echo.sqldb as sqldb

sqldb.get_records(seller, IndexType.SELLER_RESEARCH.value, condition_dict={"data_type": IndexDataType.COMPETITOR_WEBSITE_DATA.value})

In [7]:
from echo.indexing import get_vector_index
from echo.query_executor import run_perplexica_subquery, LLMSubQuery, PerplexicaSubQuery, PerplexicaSourceExtraction

vector_index = get_vector_index(seller, IndexType.BUYER_ACCOUNT_PLAN)

# def query_content(query: str, filters: MetadataFilters):
#     vector_index = get_vector_index(sub_query_inputs['seller'], sub_query.index_type)
#     metadata_filters = get_metadata_filters(sub_query.index_type, sub_query_inputs)
#     context = query_content(sub_query.query, metadata_filters)
#     response = vector_index.as_query_engine(
#         filters=filters, similarity_top_k=similarity_top_k, **kwargs
#     ).query(query)
#     return "Relevant Context:\n" + str(response)

In [22]:
buyer = 'rippling.com'

response = run_perplexica_subquery(
    PerplexicaSubQuery(
        query=f"Identify the company's key business priorities and strategic goals and pains for the company - {buyer}. "
            "Analyze 10-K reports and financial statements, leadership interviews, Earnings call transcripts and annual reports. "
            "10-K, 10-Q (SEC Filings) "
            "Also get access to hiring trends and growth of segments of the company"
            "Gain insights into the company's operations, products, services, and market position. "
            "Assess the company's revenue, profitability, and overall financial stability to gauge its potential as a client."
            "Understand the company's future plans and priorities. "
            "Identify challenges the company faces, enabling you to position your product or service as a solution to mitigate these risks. ",
        output_name="xyz"
    )
)

In [ ]:
print(response)

In [ ]:
from echo.step_templates.utilities.account_plan_creation import create_account_plan


ap_results = create_account_plan(seller=seller, buyer=buyer)

In [ ]:
from echo.query_executor import aget_query_response
from echo.query_executor import LlamaSubQuery, QueryChain, Query
from echo.step_templates.utilities.account_plan_creation import QueryTypes  
from echo.data.indexes import IndexType, IndexDataType
from echo.query_executor import PerplexicaSubQuery, PerplexicaSourceExtraction  
import asyncio
import nest_asyncio
nest_asyncio.apply()


multi_threading_team_gen = Query(
    query=(
        "You're an experienced enterprise seller. Given these company initiatives, for the ones marked relevant to the seller's product, "
        "Infer which internal team likely owns or sponsors each initiative. \n"
        "If multiple teams are involved, note primary and secondary.\n"

        "Input:\n"
        "{account_plan_value_prop}\n"

        "Return format:\n"
        "- Initiative: ...\n"
        "- Likely owning team(s): ...\n"
        "- Reasoning:\n"
    ),
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_team_gen",
)

perplexity_search_query = Query(
    query="""
    You are provided with the company initiatives and the owning team for each of them.
    
    {multi_threading_team_gen}.
    
    You are provided with the company initiatives and the relevant information about from the web about the potential team members and employees of the company that would be relevant to the company initiatives.
    Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.
    """,
    sub_queries=[
        PerplexicaSubQuery(
            query=(
                "You are provided with the company initiatives and the owning team for each of them\n"
                "{multi_threading_team_gen}\n"
                "Now, for each of the initiatives, Search for all possible employees and leaders from LinkedIn data who belong to that team and extract role, name, and background.\n"
            ),
            source_extraction_prompts=PerplexicaSourceExtraction(
                system_prompt=(
                    "You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning "
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                ),
                user_prompt=(
                    "You are provided with the company initiatives and the owning team for each of them\n"
                    "You need to extract the team members and employees of the company that would be relevant to the company initiatives.\n"
                    "Extract out the team members or employees from the below data\n"
                )
            )
        )
    ],
    output_name="perplexity_search_query"
)

# 2) do a perplexica serach here using team name and buyer name FOR EACH INITIATIVE RETURNED FROM ABOVE
# Search for all possible employees and leaders  from linkedin who belong to that team and extract role, name, and background


# 3) multi threading ROLE AND PERSON EXTRACTOR - use above response also aas input below additionally 

multi_threading_person_extractor = Query(
    query="""
        You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning 
        for every relavant initiative the company is pursuing.
        
        
        You are provided with the company initiatives and the owning team for each of them.
        {multi_threading_team_gen}.
        
        You are further provided with the relevant team members and employees of the company that would be relevant to the company initiatives as crawled from the web.
        {perplexity_search_query}
        
        Do the following:

        Given the company {buyer}, and the owning team of that initiative and the team members crawled from linkedin,
        return people who match titles commonly associated with owning this initiative.
        Focus on seniority, team fit, and tenure. Prioritize those with likely budget/influence.

        Also, Classify each as a champion, decision maker, gatekeeper and influncer within the team responsible for the initiative.
        champion - one who directly owns the pain and will want it solved
        decision maker- the one with power to purchase in the team and for the initiative
        gatekeeper - the one who will block the deal from happening or be tough to convince. This is the only role that could be outside the team like procurement , legal etc.
        influencer - the one who will influence the decision maker and champion to buy the product.

        Return:
        - initiative
        - Name
        - Title
        - Tenure
        - Team
        - Reason they likely own this initiative
        - Classification (champion, decision maker, gatekeeper, influencer) and why

        here is the list of initiatives and the owning team for each of them:
        {multi_threading_team_gen}
    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value} ## Has demo data separately
        )
    ],
    output_name="multi_threading_person_extractor"
)


# 4) multi threading outreach generator

multi_threading_outreach_generator = Query(
    query="""
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in the list
        Do the following:


        Based on this buyer's title, initiative, 
        and recent activity, seller's product details and generate a 1st outreach email that aligns to their business goals and personal context.
        You are also given similar companies the seller has helped before below.

        Tone: Crisp, consultative, relevant.

        Return:
        - Subject line
        - Message body (under 100 words)
        - CTA
        - Persona
        - Title
        - Reasoning for message


        Input:
        {multi_threading_person_extractor}
        
    """,
    sub_queries=[
        LlamaSubQuery(
            query="what is the seller's product and what pains does it solve?",
            index_type=IndexType.SELLER_RESEARCH,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="what are some companies the sellers product has helped before? be specific and metric driven",
            index_type=IndexType.SELLER_RESEARCH,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_outreach_generator"
)

account_plan_value_prop = Query(
query="""
You are a strategic sales assistant. Given a set of company initiatives and the product profile of the sellers product below as context,
identify which initiatives are *relevant* to what this product solves.

For each initiative:
- Mark as Relevant or Not Relevant
- If Relevant: explain which product capability maps to it
- If Not Relevant: explain why it's not a fit (e.g., not adjacent, unrelated)


output format:
    "initiative": "...",
    "relevant": not_relevant/ mid / highly relevant,
    "mapped_to_product": "...",
    "reasoning": "..."
    "similar buyers and their roi': "...",

""",
sub_queries=[
    # LlamaSubQuery(
    #     query="What is the details on the industry and products of the buyer account?",
    #     index_type=IndexType.BUYER_RESEARCH,
    # ),
    # LlamaSubQuery(
    #     query="What are the top 3 financial, strategic and competitive goals and pains for the buyer account to solve for?",
    #     index_type=IndexType.BUYER_RESEARCH,
    # ),
    # LlamaSubQuery(
    #     query="What are the top 3 financial priorities for the account to solve for?",
    #     index_type=IndexType.BUYER_ACCOUNT_PLAN,
    #     inputs={"query_type": QueryTypes.FMOD.value},
    # ),
    # LlamaSubQuery(
    #     query="What are the top 3 competitors that buyer might be worried about and want to tackle",
    #     index_type=IndexType.BUYER_ACCOUNT_PLAN,
    #     inputs={"query_type": QueryTypes.COMPANALYSIS.value},
    # ),
    # LlamaSubQuery(
    #     query="What is the most relevant news and recent media for the buyeraccount?",
    #     index_type=IndexType.BUYER_ACCOUNT_PLAN,
    #     inputs={"query_type": QueryTypes.RECENTNEWS.value},
    # ),
    # LlamaSubQuery(
    #     query="What are the top 3 strategic priorities for the account",
    #     index_type=IndexType.BUYER_ACCOUNT_PLAN,
    #     inputs={"query_type": QueryTypes.STRATEGY.value},
    # ),
    LlamaSubQuery(
        query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
        index_type=IndexType.SELLER_RESEARCH,
        inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
    ),
    LlamaSubQuery(
        query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
        index_type=IndexType.SELLER_RESEARCH,
        inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
    ),
    LlamaSubQuery(
        query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
        index_type=IndexType.SELLER_RESEARCH,
        inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value},
    )
],
output_name="account_plan_value_prop",
)


seller = 'https://whatfix.com/'
buyer = 'https://www.rippling.com/'


inputs = {
    "seller": seller,
    "buyer": buyer,
}

response = asyncio.run(aget_query_response(echo_query=account_plan_value_prop, inputs=inputs))
print(response)

In [ ]:
print(response[0])

In [ ]:
print(response[1])

In [4]:
multi_threading_team_gen = Query(
    query=(
        "You're an experienced enterprise seller. Given these company initiatives, for the ones marked relevant to the seller's product, "
        "Infer which internal team likely owns or sponsors each initiative. \n"
        "If multiple teams are involved, note primary and secondary.\n"

        "Input:\n"
        f"{response[0]}\n"

        "Return format:\n"
        "- Initiative: ...\n"
        "- Likely owning team(s): ...\n"
        "- Roles and titles within the team:\n"
        "- Reasoning:\n"
    ),
    sub_queries=[
        # LlamaSubQuery(
        #     query="What is the details on the industry and products of the buyer account?",
        #     index_type=IndexType.BUYER_RESEARCH,
        #     inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        # )
    ],
    output_name="multi_threading_team_gen",
)

In [ ]:
multi_response = asyncio.run(aget_query_response(echo_query=multi_threading_team_gen, inputs=inputs))
print(response)

In [ ]:
print(multi_response[0])

In [ ]:
llm_search_query = Query(
    query=(
        "You're an experienced enterprise seller. Given these company initiatives, team roles, team members and their background"
        
        "Return format:\n"
        "- Initiative: ...\n"
        "- Likely owning team(s): ...\n"
        "- Roles and titles within the team:\n"
        "- Reasoning:\n"
        "- Names of people in the team:\n"
        "- Brief background on them and their role:\n"
    ),
    sub_queries=[
        LLMSubQuery(
            query=(
                f"""
                Given these initiatives, and responsible roles for each mentioned below, find the people at these roles at {buyer}. Prioritize linkedin as the most trustable source.
                here is the input - {multi_response[0]}

                Return format:
                Keep the input as is and append three fields to them
                Name:
                Role:
                Brief overview on them and their role:
                """
            ),
            use_web_search=True
        )
    ],
    output_name="gpt4o_web_search_query"
)

perplexica_response = asyncio.run(aget_query_response(echo_query=llm_search_query, inputs=inputs))

In [ ]:
print(perplexica_response[0])

In [ ]:
multi_threading_outreach_generator = Query(
    query=f"""
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in the list
        Do the following:


        Based on this buyer's title, initiative, 
        and recent activity, seller's product details and generate a 1st outreach email that aligns to their business goals and personal context.
        You are also given similar companies the seller has helped before below.

        Tone: Crisp, consultative, relevant.

        Return:
        - Subject line
        - Message body (under 100 words)
        - CTA
        - Persona
        - Title
        - Reasoning for message


        Input:
        {perplexica_response[0]}
        
    """,
    sub_queries=[
        LlamaSubQuery(
            query="what is the seller's product and what pains does it solve?",
            index_type=IndexType.SELLER_RESEARCH,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="what are some companies the sellers product has helped before? be specific and metric driven",
            index_type=IndexType.SELLER_RESEARCH,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_outreach_generator"
)

outreach_response = asyncio.run(aget_query_response(echo_query=multi_threading_outreach_generator, inputs=inputs))

In [ ]:
print(outreach_response[0])